# Chapter 3 · Lab 3 — Measure the optimized model

Prerequisites: Lab 2's operator, block and full-model correctness; Lab 1's frozen
ranked predictions. This lab reuses [shared/performance.py](../../../shared/performance.py)
from Chapter 2 and the same Chapter 1 forward/cache path. Run top to bottom in a
fresh CUDA kernel. New outputs are matched raw observations, plotted comparisons,
speedups and explanations carried into Chapter 4 capacity/scheduling work.

In [ ]:
from pathlib import Path
import importlib, json, os, sys, time
from uuid import uuid4
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# Keep compiler disk caches from different DSL versions separate. In-memory JIT reuse remains enabled.
os.environ.setdefault('CUTE_DSL_DISABLE_FILE_CACHING', '1')
from IPython.display import display, Markdown, Image
from shared import performance as perf
study = importlib.import_module('chapters.03_kernels.code.study')
plots = importlib.import_module('chapters.03_kernels.code.plots')
RUN_GPU = True
MODEL_KEYS = ['8b', '32b']
EXTERNAL_PROFILERS = True
# Set P03_REPLAY_DIR only to inspect an existing run. Replay never creates measurements.
REPLAY_DIR = os.environ.get('P03_REPLAY_DIR')

## 1. Freeze the matched workload and predictions

Both **Qwen3-8B and Qwen3-32B**, BF16, batches **1/2/4**, prompts **128/512/2048**.
Load baseline and optimized sequentially. Each case uses **two complete warmup
runs**, **three measured repeats** and **eight decode calls after prefill**.
Checkpoint revisions, prompt and forced continuation IDs, cache conditions and
timing boundaries are identical. Saved IDs use CUDA generator seed 42. This is a
real-checkpoint GPU benchmark with synthetic IDs, not a language-quality test.

Correctness stays outside timing. Warmup traverses every growing-cache shape;
compilation during a measured repeat fails the case. Files retain predictions,
raw repeats, source hashes, flags, compiler metadata, and per-case status. OOM
cases remain explicitly unmeasured; previously successful cases are retained.
Do not run other GPU experiments concurrently with this sweep.

In [ ]:
print('Cases per model/implementation:', study.CASES)
print('Revisions:', perf.REVISIONS)
print('Timing and numerical contract:', study.manifest('preflight'))
# Freeze Amdahl inputs here using Lab 1 evidence; None means not yet supplied.
amdahl_inputs = []  # dict(model, batch, prompt, phase, baseline_ms, fraction, local_speedup, adapter_ms)
# Point these at Lab 1/2 artifacts to join frozen forecasts and profile counts.
BASELINE_DIR = Path(os.environ['P03_BASELINE_DIR']) if os.environ.get('P03_BASELINE_DIR') else None
OPTIMIZED_DIR = Path(os.environ['P03_OPTIMIZED_DIR']) if os.environ.get('P03_OPTIMIZED_DIR') else None

In [ ]:
if RUN_GPU:
    if REPLAY_DIR:
        RUN_DIR = Path(REPLAY_DIR).resolve()
        print('Replaying saved evidence; no new measurements:', RUN_DIR)
        assert (RUN_DIR/'completion.json').is_file()
    else:
        RUN_DIR = ROOT/'results'/('p03-sweep-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:6])
        study.execute(RUN_DIR, 'sweep', MODEL_KEYS, external=EXTERNAL_PROFILERS)
    raw = plots.read_rows(RUN_DIR/'results.csv')
    summary = perf.summarize_model_rows(raw)
    display(Markdown(f'Artifacts: `{RUN_DIR}` · {len(raw)} unprofiled observations'))
    display(json.loads((RUN_DIR/'status.json').read_text()))
else:
    print('GPU/checkpoint work unmeasured. Enable RUN_GPU after setup.')

## 2. Roofline and latency/throughput comparisons

The two implementations use the **same compulsory-byte intensity model** and
useful causal matmul FLOPs. These are theoretical estimates, not measured DRAM
traffic or actual executed instruction counts. Dense baseline attention does
additional masked work. The compute ceiling remains labeled as assumed.

Latency/input tok/s and latency/output tok/s are plotted against **batch**, with
separate context series. Decode aggregates all eight calls within each repeat:
`sum(output tokens) / sum(wall seconds)`; points are medians across repeats,
whiskers are observed minima/maxima, not confidence intervals.

The compact roofline uses color for model/phase, marker shape for implementation,
and marker size for batch. All saved contexts are included; `summary.csv` retains
each point’s exact workload and repeat range.

In [ ]:
if RUN_GPU:
    display(Image(filename=str(RUN_DIR/'roofline.png')))
    for key in MODEL_KEYS:
        display(Image(filename=str(RUN_DIR/f'latency_throughput_{key}.png')))

## 3. Reuse Section 2-style analytical overlays

At a fixed **2048-token prompt/prefix**, compare frozen Chapter 2 predictions,
baseline and optimized measurements: local TTFT versus prefill input throughput;
interactivity (`1/TBT` output tok/s per user) versus total decode throughput.
Use **only the first decode call** so the prefix matches the prediction. Do not
substitute the full-window mean from the previous plots. Different markers identify
implementations; the same analytical curve supports both.

In [ ]:
if RUN_GPU:
    display(Image(filename=str(RUN_DIR/'tradeoffs.png')))
    display(json.loads((RUN_DIR/'tradeoff_observations.json').read_text()))

## 4. Matched speedups and Amdahl explanations

Join by model, batch, context and phase. Baseline/candidate median latency is a
speedup even when it is below one. Extrema ratios are an observed range, not a
confidence interval or a claim of statistical significance. Do not compare one
implementation's missing/OOM case with a different workload.

When Lab 1/2 directories are supplied, `amdahl_comparison.csv` also records
instrumented CUDA span, idle intervals and attention time. These do not replace
the unprofiled latency observations. Inspect `intermediate_operator_records.csv`
for selected operator shapes, calls and inclusive allocation bytes; do not sum
nested allocation counts or interpret them as counter-measured DRAM traffic.
`profile_comparison_scope.json` records these limits and the analysis source hash.

In [ ]:
if RUN_GPU:
    matched = plots.speedups(summary)
    display(matched)
    predictions = [dict(**row, **plots.amdahl(row['baseline_ms'],row['fraction'],
                    row['local_speedup'],row.get('adapter_ms',0))) for row in amdahl_inputs]
    perf.write_json(RUN_DIR/'student_amdahl_predictions.json',predictions)
    display(predictions or 'Fill the frozen Lab 1 Amdahl inputs; no prediction has been invented.')
    explanations = [dict(model=r['model'],batch=r['batch'],prompt=r['prompt'],phase=r['phase'],
                         measured_speedup=r['speedup'],evidence=None,explanation=None) for r in matched]
    perf.write_json(RUN_DIR/'student_explanations.json',explanations)
    if BASELINE_DIR and OPTIMIZED_DIR:
        evidence = importlib.import_module('chapters.03_kernels.code.evidence')
        display(evidence.compare(BASELINE_DIR, OPTIMIZED_DIR, RUN_DIR, measurements=raw))

## Completion and handoff to Chapter 4

Complete 36 model/implementation/workload cases (972 raw forward observations),
or retain explicit per-case memory failures. Keep full-model logit checks, frozen
predictions, source/implementation hashes, flags, compilation metadata and raw
repeat data. Confirm zero unexpected fallback and no compilation during timing.
Submit roofline, sweeps, overlays, matched speedups, filled Amdahl table and a
supported explanation for one improvement or slowdown. Reproduce one equation
and identify one hypothesis the data rejected. Counters can remain explicitly
incomplete under administrator permission restrictions; required PyTorch/Nsight
Systems captures and uninstrumented measurements remain separate evidence.

[Chapter 4](../../04_runtime_and_kv/README.md) imports `OptimizedQwen3` and adds
runtime ownership/page adapters. It must preserve the documented dense cache
contract at this boundary and charge any gathers/copies to integrated latency.